# Notebook 02 — Fine-tuning LoRA (Etapa 2)

Treina o LoRA sobre o Stable Diffusion v1-5, publica no Hugging Face Hub e versiona semanticamente. Duas configurações são comparadas (Critério 3).

### 0. Montar o Drive e apontar o dataset construído no notebook 01

O dataset (imagens + metadata.jsonl) é produzido pelo notebook 01_dataset.ipynb.
Aqui apenas montamos o Drive, definimos o MESMO DATA_DIR e verificamos que está pronto.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/multimodais/dados'
os.environ['DATA_DIR'] = DATA_DIR          # exportado para a célula de treino usar via "$DATA_DIR"

meta_path = os.path.join(DATA_DIR, 'metadata.jsonl')
assert os.path.exists(meta_path), f'metadata.jsonl não encontrado em {DATA_DIR}. Rode o notebook 01_dataset.ipynb primeiro.'
meta = [json.loads(l)['file_name'] for l in open(meta_path, encoding='utf-8')]
imgs = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(('.jpg', '.png'))]
faltando = [m for m in meta if not os.path.exists(os.path.join(DATA_DIR, m))]
print(f'DATA_DIR = {DATA_DIR}')
print(f'Imagens: {len(imgs)} | entradas metadata: {len(meta)} | faltando: {len(faltando)}')
if faltando:
    print('  faltando:', faltando)
assert imgs and not faltando, 'Dataset incompleto. Rode/complete o notebook 01 antes de treinar.'

### 1. Config 1 — ambiente + treino  (rank=8, lr=1e-4, 1500 passos)

torchao NÃO é necessário para treinar LoRA e vinha quebrado (wheel py3.10 em ambiente py3.12).

In [ ]:
!pip uninstall -y diffusers torchao

# Dependências (aspas evitam o bug de redirecionamento do shell em ">=")
!pip -q install "transformers" "accelerate" "datasets" "peft"

# diffusers a partir do código-fonte (necessário para o script de exemplo)
![ -d /content/diffusers ] || git clone --depth 1 https://github.com/huggingface/diffusers
!pip -q install -e /content/diffusers
%cd /content/diffusers/examples/text_to_image
!pip -q install -r requirements.txt

# --- Checagens antes de gastar tempo ---
import os, torch
assert torch.cuda.is_available(), 'SEM GPU. Ambiente de execução -> Alterar tipo -> T4 GPU e rode de novo.'
assert os.environ.get('DATA_DIR'), 'Rode a célula #0 antes (ela define DATA_DIR).'
print('GPU:', torch.cuda.get_device_name(0), '| DATA_DIR:', os.environ['DATA_DIR'])

# --- Autenticação via Colab Secrets (🔑 -> criar HF_TOKEN de ESCRITA -> habilitar Notebook access) ---
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

# --- Treinamento CONFIG 1 (usa o MESMO DATA_DIR da célula #0 via "$DATA_DIR") ---
# IMPORTANTE: --mixed_precision='fp16' precisa ir para o accelerate E para o script.
# Sem ele no script, os pesos do LoRA não são convertidos p/ fp32 e ocorre
# "Attempting to unscale FP16 gradients."
# validation_prompt em INGLÊS (mesma língua das legendas) -> amostras fiéis ao app.
!accelerate launch \
  --num_processes=1 --num_machines=1 \
  --mixed_precision='fp16' --dynamo_backend='no' \
  train_text_to_image_lora.py \
  --pretrained_model_name_or_path='stable-diffusion-v1-5/stable-diffusion-v1-5' \
  --train_data_dir="$DATA_DIR" \
  --caption_column='text' \
  --mixed_precision='fp16' \
  --resolution=512 --random_flip \
  --train_batch_size=1 --gradient_accumulation_steps=4 \
  --max_train_steps=1500 --learning_rate=1e-4 --lr_scheduler='cosine' \
  --lr_warmup_steps=0 --rank=8 \
  --checkpointing_steps=500 --seed=42 \
  --output_dir='/content/atelie-lora' \
  --validation_prompt='estilo_lambelambe, a vintage black and white portrait of a woman at a sunday market' \
  --push_to_hub --hub_model_id='lamble-lambe/atelie'

### 2. Config 2 — treino alternativo para comparação (Critério 3: >= 2 configurações)

Diferenças vs Config 1: rank=4 (menos capacidade -> menos overfitting), lr=5e-5 (mais
conservador), 1000 passos, warmup=100. SEM --validation_prompt: evita a recarga do
pipeline no fim do treino (foi o que causou o OOM/SIGKILL na Config 1).
Pré-requisito: ter rodado a Config 1 nesta sessão (instala o ambiente e faz o %cd).

In [ ]:
import os, torch
%cd /content/diffusers/examples/text_to_image
assert torch.cuda.is_available(), 'SEM GPU. Runtime -> T4 GPU.'
assert os.environ.get('DATA_DIR'), 'Rode a célula #0 antes (define DATA_DIR).'
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

!accelerate launch \
  --num_processes=1 --num_machines=1 \
  --mixed_precision='fp16' --dynamo_backend='no' \
  train_text_to_image_lora.py \
  --pretrained_model_name_or_path='stable-diffusion-v1-5/stable-diffusion-v1-5' \
  --train_data_dir="$DATA_DIR" \
  --caption_column='text' \
  --mixed_precision='fp16' \
  --resolution=512 --random_flip \
  --train_batch_size=1 --gradient_accumulation_steps=4 \
  --max_train_steps=1000 --learning_rate=5e-5 --lr_scheduler='cosine' \
  --lr_warmup_steps=100 --rank=4 \
  --checkpointing_steps=500 --seed=42 \
  --output_dir='/content/atelie-lora-cfg2' \
  --push_to_hub --hub_model_id='lamble-lambe/atelie'

### 3. Recuperação — sobe os pesos manualmente se o push_to_hub falhar (ex.: OOM no fim do treino)

O treino salva <output_dir>/pytorch_lora_weights.safetensors ANTES do push, então mesmo
que o processo morra (SIGKILL/OOM) na validação final, os pesos estão salvos localmente.
Ajuste OUTPUT_DIR para a pasta da config que você quer salvar.

In [ ]:
import os, shutil
from huggingface_hub import HfApi
from google.colab import userdata

OUTPUT_DIR = '/content/atelie-lora'          # Config 1;  para a Config 2 use '/content/atelie-lora-cfg2'
pesos = os.path.join(OUTPUT_DIR, 'pytorch_lora_weights.safetensors')
assert os.path.exists(pesos), f'Não encontrei os pesos em {pesos}. Confira o OUTPUT_DIR.'

# 1) Backup no Drive (o /content é EFÊMERO — some ao desconectar o runtime!)
backup_dir = '/content/drive/MyDrive/Colab Notebooks/multimodais/backup_lora'
os.makedirs(backup_dir, exist_ok=True)
destino = os.path.join(backup_dir, os.path.basename(OUTPUT_DIR) + '.safetensors')
shutil.copy(pesos, destino)
print('Backup salvo no Drive:', destino)

# 2) Upload manual para o Hub (o que o push_to_hub não conseguiu concluir)
api = HfApi(token=userdata.get('HF_TOKEN'))
api.upload_file(
    path_or_fileobj=pesos,
    path_in_repo='pytorch_lora_weights.safetensors',
    repo_id='lamble-lambe/atelie',
    repo_type='model',
    commit_message=f'Upload manual dos pesos LoRA ({os.path.basename(OUTPUT_DIR)})',
)
print('Pesos enviados ao Hub. Agora rode a célula #4 (versionamento) para taguear.')

### 4. Versionamento SEMÂNTICO da versão treinada no Hugging Face Hub (MAJOR.MINOR.PATCH)

Rode após CADA config (1 e 2), para taguear as duas versões e poder comparar via revision=.
Fluxo: pergunta o tipo de mudança -> lê a versão que está EM PRODUÇÃO (maior tag semver no Hub)
-> incrementa AUTOMATICAMENTE o segmento correto -> cria a nova tag apontando p/ o último commit.

In [ ]:
import re
from huggingface_hub import HfApi
from google.colab import userdata

REPO_ID = 'lamble-lambe/atelie'
api = HfApi(token=userdata.get('HF_TOKEN'))

SEMVER = re.compile(r'^v?(\d+)\.(\d+)\.(\d+)$')

def versao_producao(repo_id):
    """Maior tag semver do repo = versão em produção (retorna (0,0,0) se não houver)."""
    try:
        tags = [t.name for t in api.list_repo_refs(repo_id).tags]
    except Exception:
        tags = []
    versoes = [tuple(map(int, m.groups())) for t in tags if (m := SEMVER.match(t))]
    return max(versoes) if versoes else (0, 0, 0)

def proxima_versao(atual, tipo):
    maj, mnr, pat = atual
    if tipo == 3:  return (maj + 1, 0, 0)       # BREAKING -> incrementa MAJOR, zera o resto
    if tipo == 2:  return (maj, mnr + 1, 0)     # FEATURE  -> incrementa MINOR, zera PATCH
    if tipo == 1:  return (maj, mnr, pat + 1)   # BUG      -> incrementa PATCH
    raise ValueError('Opção inválida (use 1, 2 ou 3).')

# 1) versão atual em produção
atual = versao_producao(REPO_ID)
print(f'Versão em produção no Hub: v{atual[0]}.{atual[1]}.{atual[2]}\n')

# 2) pergunta o tipo de mudança
print('Que tipo de mudança este treino representa?')
print('  1 = BUG       (correção)  -> PATCH   x.x.(+1)')
print('  2 = FEATURE   (melhoria)  -> MINOR   x.(+1).0')
print('  3 = BREAKING  (ruptura)   -> MAJOR   (+1).0.0')
tipo = int(input('Digite o número correspondente (1/2/3): ').strip())

# 3) calcula a nova versão e cria a tag
nova = proxima_versao(atual, tipo)
TAG = f'v{nova[0]}.{nova[1]}.{nova[2]}'
rotulo = {1: 'BUG/patch', 2: 'FEATURE/minor', 3: 'BREAKING/major'}[tipo]
desc = input(f'Descrição da {TAG} (ex.: "rank=4, lr=5e-5, 1000 passos") [enter p/ padrão]: ').strip()
DESCRICAO = desc or f'{rotulo} — Ateliê Generativo (estilo lambe-lambe)'

print(f'\nProdução v{atual[0]}.{atual[1]}.{atual[2]}  --({rotulo})-->  {TAG}')
api.create_tag(repo_id=REPO_ID, tag=TAG, tag_message=DESCRICAO)
print(f'✅ Nova versão publicada: {TAG}')
print(f'   Hub:      https://huggingface.co/{REPO_ID}/tree/{TAG}')
print(f'   Carregar: pipe.load_lora_weights("{REPO_ID}", revision="{TAG}")')

## Comparação das configurações (Critério 3)

Treinamos **duas configurações** e taguamos cada uma no Hub. Registre no relatório:

| Hiperparâmetro | Config 1 | Config 2 |
|---|---|---|
| `rank` | 8 | 4 |
| `learning_rate` | 1e-4 | 5e-5 |
| `max_train_steps` | 1500 | 1000 |
| `lr_warmup_steps` | 0 | 100 |
| tag no Hub | `v_._._` | `v_._._` |

**O que analisar e justificar** (o barema avalia a justificativa, não valores decorados):

- **Fidelidade ao estilo** (P&B / vintage): gere os mesmos prompts com cada versão via `revision=` e compare.
- **Overfitting / memorização**: a Config 1 (rank 8, 1500 passos ≈ 137 épocas em 41 imagens) tende a **memorizar** mais; a Config 2 (rank 4, 1000 passos, warmup) tende a **generalizar** melhor. Confirme com a verificação de memorização do notebook 03.
- **Escolha final para produção**: qual versão você promoveu (`LORA_REVISION` no Space) e **por quê**.

> A grade comparativa 6×2, o CLIPScore e a avaliação humana ficam no **notebook 03**.